##### generates D:\MICB_Projects\8_aml_detective\agent_components\agents_personas.json
##### used in the main flow to generate hyde documents

In [25]:
# Standard Library
import os
import re
import sys
import copy
import json
import time
import asyncio
import operator
import threading
from datetime import datetime
from typing import TypedDict, Annotated, List, Dict, Optional, Set, Literal, Callable, get_args
from urllib.parse import urlparse
from collections import Counter

# Environment
from dotenv import load_dotenv

# OpenAI
from openai import OpenAI

# LangChain
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import BaseMessage, AnyMessage, ToolMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.language_models import BaseChatModel
from langchain_core.tools import tool, StructuredTool, InjectedToolArg
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages.utils import count_tokens_approximately

# LangGraph
from langgraph.graph import add_messages, START, END, StateGraph
from langgraph.checkpoint.memory import MemorySaver

# Pydantic
from pydantic import BaseModel, Field

# Search
from googleapiclient.discovery import build
from tavily import TavilyClient
from serpapi import GoogleSearch

# ML
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import numpy.typing as npt

# Notebook
import subprocess
from IPython.display import Image, display

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [26]:
import logging

LOG_FILENAME = "aml_research.log"

# Configure logger
logger = logging.getLogger("aml_research")
logger.setLevel(logging.DEBUG)

# File handler — mode='w' clears the file on each run
file_handler = logging.FileHandler(LOG_FILENAME, mode='w', encoding="utf-8")
file_handler.setLevel(logging.DEBUG)

# Console handler — shows INFO and above
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)

# Formatter
formatter = logging.Formatter(
    "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
file_handler.setFormatter(formatter)
console_handler.setFormatter(formatter)

logger.addHandler(file_handler)
logger.addHandler(console_handler)

logger.info(f"Logger initialized — writing to {LOG_FILENAME}")


2026-04-27 15:50:02 | INFO     | aml_research | Logger initialized — writing to aml_research.log
2026-04-27 15:50:02 | INFO     | aml_research | Logger initialized — writing to aml_research.log


In [27]:
# Prompts
from agent_components.prompts_v2 import (
    extract_evidence_claims_prompt,
    expert_instructions,
    url_summary_instructions,
    system_messages_search_tools,
    final_summary_prompt
)

# States & Pydantic models
from agent_components.states_v2 import (
    Journalist, HydePerspectives,
    AllowedClaimType, EvidenceClaim, ClaimsFromSummaries,
    ScenarioPool, Scenario_Selected,
    LinkCollection, AllowedSeverityLevel, ContentSummary,
    QueryComponentsInState,
    DOMAIN_EXCLUDE,
    AllowedEvidenceAssessment, AssessEvidenceQuality,
    merge_links_by_url, merge_query_data_by_id,
    UnifiedResearchState, QueryPerformance,
    FinalReport
)

# Query components
from agent_components.query_components import QueryComponentsCalc

from agent_components.dictionary import topic_keywords

## import utilities
from agent_components.utils import ( APIVault, first_n_words ,  list_to_string , count_leaves)


In [28]:
## populate apis
key_vault = APIVault()
key_vault.add_key("openai_key", os.environ.get("OPENAI_API_KEY"))
key_vault.add_key("tavily_key", os.environ.get("TAVILY_API_KEY"))
key_vault.add_key("google_key", os.environ.get("GOOGLE_API_KEY"))
key_vault.add_key("serp_google_key", os.environ.get("SERP_GOOGLE_API_KEY"))


#### Define LLM Engines

In [29]:


# Journalist persona generation
llm_expert_generation = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.3,
    max_tokens=1500,
    timeout=60,
    max_retries=3,
    api_key=key_vault.get_key("openai_key")
)

# HyDE article generation (creative variation per journalist style)
llm_hyde_generation = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.5,
    max_tokens=600,
    timeout=60,
    max_retries=3,
    api_key=key_vault.get_key("openai_key")
)

# Name variations generation (pure extraction, near-zero temperature)
llm_names_variation = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.0,
    max_tokens=50,
    timeout=30,
    max_retries=2,
    api_key=key_vault.get_key("openai_key")
)



#### Variations of company names

In [30]:
def generate_name_regex(entity_name: str, llm_instance: BaseChatModel) -> tuple[list[str], re.Pattern]:
    """
    Generate name variations using LLM and compile regex pattern for matching.
    
    Uses LLM to create multiple name variants (original, transliterated Russian, 
    versions without spaces) and builds a case-insensitive regex to match any variant.
    
    Args:
        entity_name: Company or entity name to generate variations for
        llm_instance: Language model instance for generating name variations
        
    Returns:
        tuple containing:
            - entity_names_variations (list[str]): List of name variant strings
            - names_search_regexp (re.Pattern): Compiled regex pattern matching any variant
            
    Raises:
        RuntimeError: If LLM invocation fails or returns invalid response
        ValueError: If no valid name variations can be generated
        
    Example:
        >>> from langchain_openai import ChatOpenAI
        >>> llm = ChatOpenAI(model="gpt-4")
        >>> variations, regex = generate_name_regex("Acme Corp", llm)
        >>> print(variations)
        ['Acme Corp', 'Акме Корп', 'AcmeCorp', 'АкмеКорп']
        >>> regex.search("Looking for acmecorp info")
        <re.Match object...>
    """
    prompt = f"""
    You are an expert at generating a single-line string of company name variants separated by '|'.

    OUTPUT FORMAT (choose exactly one based on spaces in the Original name):
    - If the Original name CONTAINS whitespace:
    <ORIGINAL_NAME>|<RUSSIAN_NAME>|<ORIGINAL_NO_SPACES>|<RUSSIAN_NO_SPACES>
    - If the Original name DOES NOT CONTAIN whitespace:
    <ORIGINAL_NAME>|<RUSSIAN_NAME>

    Rules:
    - ORIGINAL_NAME = the input company name verbatim (do not modify punctuation, casing, or suffixes).
    - RUSSIAN_NAME = a **transliteration/phonetic rewrite** of ORIGINAL_NAME into Cyrillic. **Do NOT translate any words** (brand words, common nouns, legal suffixes, country names, etc.). Only rewrite letters to approximate English pronunciation. If the input is already Cyrillic/Russian, repeat it verbatim. If unsure, copy the original as is.
    - ORIGINAL_NO_SPACES = ORIGINAL_NAME with ALL whitespace removed (preserve punctuation/casing). Include ONLY if the Original name contains whitespace.
    - RUSSIAN_NO_SPACES = RUSSIAN_NAME with ALL whitespace removed (preserve punctuation/casing). Include ONLY if the Original name contains whitespace.
    - Preserve hyphens, punctuation, and casing exactly where they appear in ORIGINAL_NAME.
    - Output MUST be exactly one line, no leading/trailing spaces, no extra spaces around '|', no quotes, no notes, no extra variants, no newlines.

    Original name: {entity_name}
    """.strip()

    try:
        resp = llm_instance.invoke(prompt)
        pipe_line = (getattr(resp, "content", "") or "").strip()

        if not pipe_line:
            raise ValueError("LLM returned empty response")

        # Parse "Original | Russian | ...", fallback to original if needed
        parts = [p.strip() for p in pipe_line.split("|")]

        if not parts:
            raise ValueError("No valid name variations generated")
            
        # Compile strict, case-insensitive pattern
        pattern_str = r"(?:%s)" % "|".join(map(re.escape, parts))
        name_re = re.compile(pattern_str, re.IGNORECASE)

        return parts, name_re
    
    except Exception as e:
        raise RuntimeError(f"Name variation generation failed for '{entity_name}': {e}") from e

In [31]:
def extract_domain(url:str) -> str: 
    parsed = urlparse(url)
    displayLink = f"{parsed.scheme}://{parsed.netloc}"
    return displayLink

#### Architecture

In [32]:
# Prepare input for search queries
parts, name_re = generate_name_regex(entity_name="Lukoil", llm_instance=llm_names_variation)
qc = QueryComponentsCalc("Lukoil", DOMAIN_EXCLUDE, parts)
qc.build_all()
query_state = qc.to_state()

# inject regex pattern into state
query_state = query_state.model_copy(update={"names_search_regexp": name_re.pattern})


#### Generate Personas

In [33]:
# Flatten all English topic phrases from query_state
all_topics = []
for topic in query_state.search_topics:
    for lang, phrases in query_state.search_topics[topic].items():
        if lang == "en":
            all_topics.extend(phrases)

print(all_topics)

['money laundering', 'fraud', 'tax evasion', 'embezzlement', 'financial crime', 'crime', 'corruption', 'bribery', 'influence peddling', 'kickback', 'illicit enrichment', 'organized crime', 'mafia', 'cartel', 'smuggling', 'trafficking', 'sanctions', 'blacklist', 'OFAC', 'penalty', 'regulatory action', 'license revoked', 'lawsuit', 'indictment', 'arrest', 'conviction', 'criminal charges', 'investigation']


In [34]:
max_journalists = 20

structured_llm = (llm_expert_generation
                    .with_structured_output(HydePerspectives)
                    .with_retry(stop_after_attempt=3))

journalists_by_topic = {}

for topic_name in query_state.search_topics:

    system_message = expert_instructions.format(
        crime_topic=topic_name,
        max_journalists=max_journalists
    )

    existing_summary = []
    for t, journalists in journalists_by_topic.items():
        for j in journalists:
            existing_summary.append(f"- [{t}] {j.expertise} | {j.style}")

    existing_text = "\n".join(existing_summary) if existing_summary else "None yet."

    human_message = f"""Generate the set of journalists according to instructions.

Already created personas (DO NOT repeat these expertise/style combinations):
{existing_text}

Ensure each new journalist has a DISTINCT expertise and style from the ones listed above."""

    result = structured_llm.invoke([
        SystemMessage(content=system_message),
        HumanMessage(content=human_message)
    ])

    journalists_by_topic[topic_name] = result.journalists

In [35]:
journalists_by_topic

{'financial': [Journalist(expertise='Regulatory enforcement breach analysis', perspective='Prioritizes unresolved regulatory sanctions and frequency of enforcement actions over voluntary disclosures when assessing risk.', style='forensic, regulation-centric, meticulous, compliance-focused, precedent-aware'),
  Journalist(expertise='Transactional anomaly detection in correspondent banking', perspective='Focuses on unusual transaction patterns and volume spikes inconsistent with client profiles as primary risk indicators.', style='data-driven, pattern-focused, algorithmic, detail-oriented, anomaly-sensitive'),
  Journalist(expertise='Third-party vendor risk and governance failures', perspective='Emphasizes lapses in vendor due diligence and control breakdowns in outsourced operations as key risk factors.', style='operational, governance-aware, pragmatic, control-centric, vendor-focused'),
  Journalist(expertise='Media sentiment and controversy longevity tracking', perspective='Weights pe

In [36]:
import json
from pathlib import Path

output_path = Path(r"agent_components\agents_personas.json")

# Serialize: convert Journalist objects to dicts
personas_serializable = {
    topic: [j.model_dump() for j in journalists]
    for topic, journalists in journalists_by_topic.items()
}

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(personas_serializable, f, ensure_ascii=False, indent=2)

print(f"Saved → {output_path}")

Saved → agent_components\agents_personas.json


#### Now generate articles

In [37]:
# now agents personalities
with open("agent_components/agents_personas.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)

journalists_by_topic = {
    topic: [Journalist(**j) for j in journalists]
    for topic, journalists in loaded.items()
}

In [38]:
def generate_hyde_articles_prebuilt(
    journalists_by_topic: dict
):
    """
    Pre-generate HyDE articles for all topics and personas using a <|company_name|> placeholder.

    Iterates over all topics and their journalist personas in journalists_by_topic,
    generates one hypothetical article per persona.
    Articles use <|company_name|> as a placeholder to be replaced at runtime before embedding.

    Args:
        journalists_by_topic: Dict[str, List[Journalist]] — pre-loaded personas keyed by topic

    Returns:
        Dict[str, List[str]] — articles keyed by topic name
    """

    article_template = """You are a journalist with the profile below. Write a short, hypothetical article for retrieval (HyDE).
            
            Company name (repeat it at least 3 times): <|company_name|>
            Expertise: {expertise}
            Perspective: {perspective}
            Style: {style}
            Topic to cover: {crime_topic}

            Constraints:
            - 350 - 400 words, plain text (no markdown).
            - Use <|company_name|> as the company name throughout — repeat it at least 3 times.
            - Include relevant lexicon where natural: adverse media, negative news, reputational risk,
              anti-money laundering (AML), know-your-customer (KYC), transaction monitoring, suspicious
              activity reports (SARs), sanctions, beneficial ownership, PEP, consent order, settlement, probe,
              enforcement action, regulatory fine, criminal investigation.
            - Keep a neutral, reportorial tone; this is hypothetical for search, not a claim.
            - If topic lacks dates/amounts, use vague ranges. Do not fabricate concrete numbers.

            Structure to follow (inline, no headings):
            - Headline that includes "<|company_name|>".
            - One-sentence dek summarizing the angle.
            - Lede paragraph naming <|company_name|> and the jurisdiction/sector if implied by the topic.
            - One paragraph with AML/KYC process vocabulary and typical risk indicators.
            - One paragraph on regulatory/enforcement posture (investigation/settlement/fine allegations as applicable).
            - Closing sentence that restates <|company_name|> and the core issue.

            Write the article now."""

    hyde_articles_by_topic = {}

    for topic_name, journalists in journalists_by_topic.items():
        hyde_articles_by_topic[topic_name] = []

        for elem in journalists:
            prompt = article_template.format(
                expertise=elem.expertise,
                perspective=elem.perspective,
                style=elem.style,
                crime_topic=topic_name
            )

            correction = ""  # empty by default

            for attempt in range(3):
                full_prompt = prompt + correction
                article = llm_hyde_generation.invoke([HumanMessage(content=full_prompt)])

                if "<|company_name|>" in article.content:
                    correction = ""  # reset after fix
                    break
                else:
                    print("ERROR, Missing placeholder")
                    correction = "\n\nCRITICAL: You must use <|company_name|> as the company name placeholder. Rewrite the article using <|company_name|> throughout."

            hyde_articles_by_topic[topic_name].append(article.content)

        print(f"[generate_hyde_articles_prebuilt] '{topic_name}' → {len(hyde_articles_by_topic[topic_name])} articles")

    return hyde_articles_by_topic



In [39]:

# run it
hyde_articles_by_topic = generate_hyde_articles_prebuilt(journalists_by_topic)

with open(r"D:\MICB_Projects\8_aml_detective\agent_components\agents_hyde_articles.json", "w", encoding="utf-8") as f:
    json.dump(hyde_articles_by_topic, f, ensure_ascii=False, indent=2)

print(f"Saved → agents_hyde_articles.json")

[generate_hyde_articles_prebuilt] 'financial' → 20 articles
[generate_hyde_articles_prebuilt] 'corruption' → 20 articles
[generate_hyde_articles_prebuilt] 'organized_crime' → 20 articles
[generate_hyde_articles_prebuilt] 'sanctions' → 20 articles
[generate_hyde_articles_prebuilt] 'legal' → 20 articles
Saved → agents_hyde_articles.json
